# 从PCA到自编码器：降维的原理与实践

本notebook探索各种降维技术，从传统的PCA到深度学习中的自编码器。我们将使用MNIST数据集作为实验对象，比较不同方法的效果。

## 1. 数据准备与预处理

首先导入必要的库，并加载MNIST数据集进行预处理。

In [ ]:
# 导入必要的库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import seaborn as sns

# 设置随机种子确保结果可复现
np.random.seed(42)
torch.manual_seed(42)

# 设置matplotlib参数
plt.rcParams['figure.figsize'] = (12, 8)
plt.style.use('ggplot')

In [ ]:
# 加载MNIST数据集
transform = transforms.Compose([transforms.ToTensor()])

# 训练集
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 测试集
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 处理数据以便用于非深度学习方法
X_train = train_dataset.data.numpy().reshape(len(train_dataset.data), -1) / 255.0
y_train = train_dataset.targets.numpy()

X_test = test_dataset.data.numpy().reshape(len(test_dataset.data), -1) / 255.0
y_test = test_dataset.targets.numpy()

# 数据标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"训练数据形状: {X_train.shape}")
print(f"测试数据形状: {X_test.shape}")

## 2. PCA原理与实现

主成分分析(PCA)是一种线性降维技术，通过找到具有最大方差的正交投影来减少数据的维度。下面我们从头实现PCA，然后使用sklearn库进行比较。

In [ ]:
def pca_from_scratch(X, n_components):
    """
    从头实现PCA算法
    
    参数:
    X: 输入数据，形状为 (n_samples, n_features)
    n_components: 要保留的主成分数量
    
    返回:
    X_pca: 降维后的数据，形状为 (n_samples, n_components)
    components: 主成分向量，形状为 (n_components, n_features)
    explained_variance_ratio: 解释的方差比例
    """
    # 对数据进行中心化
    X_centered = X - np.mean(X, axis=0)
    
    # 计算协方差矩阵
    cov_matrix = np.cov(X_centered, rowvar=False)
    
    # 计算协方差矩阵的特征值和特征向量
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    
    # 特征值和特征向量按照特征值从大到小排序
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # 选择前n_components个特征向量
    components = eigenvectors[:, :n_components]
    
    # 计算解释的方差比例
    explained_variance_ratio = eigenvalues[:n_components] / np.sum(eigenvalues)
    
    # 将数据投影到主成分上
    X_pca = np.dot(X_centered, components)
    
    return X_pca, components, explained_variance_ratio

# 使用自定义PCA函数
n_components = 30
X_pca_custom, components, explained_variance_ratio = pca_from_scratch(X_train_scaled, n_components)

# 使用sklearn的PCA
pca = PCA(n_components=n_components)
X_pca_sklearn = pca.fit_transform(X_train_scaled)

# 比较解释的方差比例
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_components + 1), np.cumsum(explained_variance_ratio), marker='o', linestyle='-', color='b', label='自定义PCA')
plt.plot(range(1, n_components + 1), np.cumsum(pca.explained_variance_ratio_), marker='s', linestyle='-', color='r', label='sklearn PCA')
plt.xlabel('主成分数量')
plt.ylabel('累积解释方差比例')
plt.title('PCA累积解释方差')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# 选择2个主成分进行可视化
pca_viz = PCA(n_components=2)
X_pca_2d = pca_viz.fit_transform(X_train_scaled)

# 绘制散点图
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_train, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter, label='数字类别')
plt.title('使用PCA降至2维的MNIST数据可视化')
plt.xlabel('主成分1')
plt.ylabel('主成分2')
plt.grid(True)
plt.show()

In [ ]:
# 探索不同主成分数量下的重构效果
def reconstruct_from_pca(X, pca_model, n_components):
    """使用PCA模型重构原始数据"""
    X_pca = pca_model.transform(X)
    X_reconstructed = np.dot(X_pca[:, :n_components], pca_model.components_[:n_components, :]) + pca_model.mean_
    return X_reconstructed

# 训练完整的PCA模型
full_pca = PCA(n_components=X_train_scaled.shape[1])
full_pca.fit(X_train_scaled)

# 不同主成分数量
components_to_test = [5, 10, 20, 50, 100, 200]

plt.figure(figsize=(15, 8))
for i, n in enumerate(components_to_test):
    # 重构数据
    X_reconstructed = reconstruct_from_pca(X_test_scaled, full_pca, n)
    
    # 计算重构误差
    mse = np.mean((X_test_scaled - X_reconstructed) ** 2)
    
    # 显示原始图像和重构图像
    plt.subplot(2, len(components_to_test), i + 1)
    plt.imshow(X_test[0].reshape(28, 28), cmap='gray')
    plt.title('原始图像')
    plt.axis('off')
    
    plt.subplot(2, len(components_to_test), i + 1 + len(components_to_test))
    plt.imshow(scaler.inverse_transform(X_reconstructed)[0].reshape(28, 28), cmap='gray')
    plt.title(f'{n}个主成分\nMSE: {mse:.4f}')
    plt.axis('off')

plt.suptitle('不同主成分数量下的图像重构效果', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

## 3. t-SNE降维可视化

t-SNE(t-distributed Stochastic Neighbor Embedding)是一种非线性降维技术，特别适合可视化高维数据。它与PCA不同，t-SNE主要关注保持数据点之间的局部关系。

In [ ]:
# 使用t-SNE降维到2D进行可视化
# 注意：t-SNE计算开销较大，这里只使用一部分数据
sample_size = 5000
random_indices = np.random.choice(X_train_scaled.shape[0], sample_size, replace=False)
X_sample = X_train_scaled[random_indices]
y_sample = y_train[random_indices]

# 应用t-SNE
print("正在应用t-SNE，这可能需要几分钟...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_sample)

# 绘制t-SNE结果
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sample, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, label='数字类别')
plt.title('使用t-SNE降维的MNIST数据可视化')
plt.xlabel('t-SNE维度1')
plt.ylabel('t-SNE维度2')
plt.grid(True)
plt.show()

In [ ]:
# 比较PCA和t-SNE在相同样本上的表现
pca_2d = PCA(n_components=2)
X_pca_sample = pca_2d.fit_transform(X_sample)

plt.figure(figsize=(18, 8))

# PCA结果
plt.subplot(1, 2, 1)
scatter1 = plt.scatter(X_pca_sample[:, 0], X_pca_sample[:, 1], c=y_sample, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter1, label='数字类别')
plt.title('PCA降维 (2D)')
plt.xlabel('主成分1')
plt.ylabel('主成分2')
plt.grid(True)

# t-SNE结果
plt.subplot(1, 2, 2)
scatter2 = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sample, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter2, label='数字类别')
plt.title('t-SNE降维 (2D)')
plt.xlabel('t-SNE维度1')
plt.ylabel('t-SNE维度2')
plt.grid(True)

plt.suptitle('PCA vs t-SNE降维比较', fontsize=16)
plt.tight_layout()
plt.show()

## 4. 使用PyTorch构建自编码器

自编码器是一种神经网络架构，用于无监督学习，它学习将输入数据压缩到低维潜在空间，然后尝试从这个压缩表示中重建原始输入。

In [ ]:
# 定义一个基础自编码器模型
class BasicAutoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim):
        super(BasicAutoencoder, self).__init__()
        
        # 编码器
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, encoding_dim),
            nn.ReLU()
        )
        
        # 解码器
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid()  # 使用Sigmoid将输出限制在[0,1]范围内，符合MNIST像素值
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded
    
    def encode(self, x):
        return self.encoder(x)

# 定义一个栈式自编码器模型
class StackedAutoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dims):
        super(StackedAutoencoder, self).__init__()
        
        # 编码器 - 逐层减小维度
        encoder_layers = []
        prev_dim = input_dim
        
        for dim in encoding_dims[:-1]:
            encoder_layers.append(nn.Linear(prev_dim, dim))
            encoder_layers.append(nn.ReLU())
            prev_dim = dim
            
        encoder_layers.append(nn.Linear(prev_dim, encoding_dims[-1]))
        encoder_layers.append(nn.ReLU())
        
        self.encoder = nn.Sequential(*encoder_layers)
        
        # 解码器 - 逐层增加维度
        decoder_layers = []
        prev_dim = encoding_dims[-1]
        
        for dim in reversed(encoding_dims[:-1]):
            decoder_layers.append(nn.Linear(prev_dim, dim))
            decoder_layers.append(nn.ReLU())
            prev_dim = dim
            
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        decoder_layers.append(nn.Sigmoid())
        
        self.decoder = nn.Sequential(*decoder_layers)
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded
    
    def encode(self, x):
        return self.encoder(x)

# 定义一个去噪自编码器模型
class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim, noise_factor=0.3):
        super(DenoisingAutoencoder, self).__init__()
        
        self.noise_factor = noise_factor
        
        # 编码器
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, encoding_dim),
            nn.ReLU()
        )
        
        # 解码器
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid()
        )
    
    def add_noise(self, x):
        noise = torch.randn_like(x) * self.noise_factor
        noisy_x = x + noise
        # 裁剪值到[0,1]范围
        return torch.clamp(noisy_x, 0., 1.)
    
    def forward(self, x):
        noisy_x = self.add_noise(x)
        encoded = self.encoder(noisy_x)
        decoded = self.decoder(encoded)
        return encoded, decoded, noisy_x
    
    def encode(self, x):
        return self.encoder(x)

## 5. 自编码器训练与评估

下面我们将训练不同类型的自编码器模型，并评估它们的重构性能。

In [ ]:
# 准备PyTorch数据集
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)

train_dataset = TensorDataset(X_train_tensor, X_train_tensor)  # 输入=目标
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, X_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# 设置设备(GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 训练函数
def train_autoencoder(model, train_loader, test_loader, epochs=10, lr=0.001):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_losses = []
    test_losses = []
    
    for epoch in range(epochs):
        # 训练阶段
        model.train()
        train_loss = 0
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()
            
            # 处理不同类型的自编码器
            if isinstance(model, DenoisingAutoencoder):
                _, output, noisy_data = model(data)
                loss = criterion(output, data)  # 重构干净的数据
            else:
                _, output = model(data)
                loss = criterion(output, data)
                
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # 测试阶段
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for batch_idx, (data, _) in enumerate(test_loader):
                data = data.to(device)
                
                if isinstance(model, DenoisingAutoencoder):
                    _, output, _ = model(data)
                else:
                    _, output = model(data)
                    
                loss = criterion(output, data)
                test_loss += loss.item()
                
        test_loss /= len(test_loader)
        test_losses.append(test_loss)
        
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.6f}, Test Loss: {test_loss:.6f}")
    
    return train_losses, test_losses

# 可视化重构结果
def visualize_reconstructions(model, data_loader, num_examples=10):
    model.eval()
    with torch.no_grad():
        for batch_idx, (data, _) in enumerate(data_loader):
            data = data.to(device)
            
            if isinstance(model, DenoisingAutoencoder):
                _, reconstructions, noisy_data = model(data)
                noisy_data = noisy_data.cpu().numpy()
            else:
                _, reconstructions = model(data)
                noisy_data = None
                
            data = data.cpu().numpy()
            reconstructions = reconstructions.cpu().numpy()
            
            break  # 只取一个batch
    
    plt.figure(figsize=(20, 4))
    
    if noisy_data is not None:
        # 显示原始图像、加噪图像和重构图像
        for i in range(num_examples):
            # 原始图像
            ax = plt.subplot(3, num_examples, i + 1)
            plt.imshow(data[i].reshape(28, 28), cmap='gray')
            plt.title("原始图像")
            plt.axis('off')
            
            # 加噪图像
            ax = plt.subplot(3, num_examples, i + 1 + num_examples)
            plt.imshow(noisy_data[i].reshape(28, 28), cmap='gray')
            plt.title("加噪图像")
            plt.axis('off')
            
            # 重构图像
            ax = plt.subplot(3, num_examples, i + 1 + 2*num_examples)
            plt.imshow(reconstructions[i].reshape(28, 28), cmap='gray')
            plt.title("重构图像")
            plt.axis('off')
    else:
        # 显示原始图像和重构图像
        for i in range(num_examples):
            # 原始图像
            ax = plt.subplot(2, num_examples, i + 1)
            plt.imshow(data[i].reshape(28, 28), cmap='gray')
            plt.title("原始图像")
            plt.axis('off')
            
            # 重构图像
            ax = plt.subplot(2, num_examples, i + 1 + num_examples)
            plt.imshow(reconstructions[i].reshape(28, 28), cmap='gray')
            plt.title("重构图像")
            plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# 训练基础自编码器
input_dim = X_train.shape[1]  # 784 for MNIST
encoding_dim = 32  # 潜在空间维度

print("训练基础自编码器...")
basic_autoencoder = BasicAutoencoder(input_dim, encoding_dim)
basic_train_losses, basic_test_losses = train_autoencoder(basic_autoencoder, train_loader, test_loader, epochs=10)

# 可视化基础自编码器的重构结果
visualize_reconstructions(basic_autoencoder, test_loader)

In [ ]:
# 训练栈式自编码器
encoding_dims = [512, 256, 128, 32]  # 逐层减小的维度

print("训练栈式自编码器...")
stacked_autoencoder = StackedAutoencoder(input_dim, encoding_dims)
stacked_train_losses, stacked_test_losses = train_autoencoder(stacked_autoencoder, train_loader, test_loader, epochs=10)

# 可视化栈式自编码器的重构结果
visualize_reconstructions(stacked_autoencoder, test_loader)

In [ ]:
# 训练去噪自编码器
noise_factor = 0.3  # 噪声系数

print("训练去噪自编码器...")
denoising_autoencoder = DenoisingAutoencoder(input_dim, encoding_dim, noise_factor)
denoising_train_losses, denoising_test_losses = train_autoencoder(denoising_autoencoder, train_loader, test_loader, epochs=10)

# 可视化去噪自编码器的重构结果
visualize_reconstructions(denoising_autoencoder, test_loader)

In [ ]:
# 比较不同自编码器模型的训练过程
plt.figure(figsize=(12, 5))

# 训练损失
plt.subplot(1, 2, 1)
plt.plot(basic_train_losses, label='基础自编码器')
plt.plot(stacked_train_losses, label='栈式自编码器')
plt.plot(denoising_train_losses, label='去噪自编码器')
plt.title('训练损失')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)

# 测试损失
plt.subplot(1, 2, 2)
plt.plot(basic_test_losses, label='基础自编码器')
plt.plot(stacked_test_losses, label='栈式自编码器')
plt.plot(denoising_test_losses, label='去噪自编码器')
plt.title('测试损失')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 6. 降维效果比较与可视化

最后，我们比较PCA、t-SNE和自编码器的降维效果。

In [ ]:
# 从各种方法获取2D降维结果

# 1. 使用PCA降至2维
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_test_scaled[:5000])  # 使用5000个样本

# 2. 使用t-SNE降至2维
print("正在计算t-SNE降维结果...")
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne_2d = tsne_2d.fit_transform(X_test_scaled[:5000])

# 3. 使用自编码器降至2维
# 首先定义一个2维输出的自编码器
autoencoder_2d = BasicAutoencoder(input_dim, encoding_dim=2)
autoencoder_2d = autoencoder_2d.to(device)

# 训练自编码器
train_dataset_2d = TensorDataset(X_train_tensor[:5000], X_train_tensor[:5000])
train_loader_2d = DataLoader(train_dataset_2d, batch_size=128, shuffle=True)

test_dataset_2d = TensorDataset(X_test_tensor[:5000], X_test_tensor[:5000])
test_loader_2d = DataLoader(test_dataset_2d, batch_size=128, shuffle=False)

print("训练2D自编码器...")
_, _ = train_autoencoder(autoencoder_2d, train_loader_2d, test_loader_2d, epochs=10, lr=0.001)

# 使用自编码器获取2D表示
autoencoder_2d.eval()
with torch.no_grad():
    X_ae_2d = autoencoder_2d.encode(X_test_tensor[:5000].to(device)).cpu().numpy()

y_test_subset = y_test[:5000]

# 可视化比较
plt.figure(figsize=(18, 6))

# PCA结果
plt.subplot(1, 3, 1)
scatter1 = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_test_subset, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter1, label='数字类别')
plt.title('PCA (2D)')
plt.xlabel('维度1')
plt.ylabel('维度2')
plt.grid(True)

# t-SNE结果
plt.subplot(1, 3, 2)
scatter2 = plt.scatter(X_tsne_2d[:, 0], X_tsne_2d[:, 1], c=y_test_subset, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter2, label='数字类别')
plt.title('t-SNE (2D)')
plt.xlabel('维度1')
plt.ylabel('维度2')
plt.grid(True)

# 自编码器结果
plt.subplot(1, 3, 3)
scatter3 = plt.scatter(X_ae_2d[:, 0], X_ae_2d[:, 1], c=y_test_subset, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter3, label='数字类别')
plt.title('自编码器 (2D)')
plt.xlabel('维度1')
plt.ylabel('维度2')
plt.grid(True)

plt.suptitle('PCA vs t-SNE vs 自编码器降维比较', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 比较不同方法的重构性能

# 1. PCA重构
pca_32 = PCA(n_components=32)
pca_32.fit(X_train_scaled)
X_pca_test = pca_32.transform(X_test_scaled[:100])
X_pca_reconstructed = pca_32.inverse_transform(X_pca_test)

# 2. 自编码器重构
basic_autoencoder.eval()
with torch.no_grad():
    _, reconstructions = basic_autoencoder(X_test_tensor[:100].to(device))
    X_ae_reconstructed = reconstructions.cpu().numpy()

# 计算MSE
pca_mse = np.mean((X_test_scaled[:100] - X_pca_reconstructed) ** 2)
ae_mse = np.mean((X_test[:100] - X_ae_reconstructed) ** 2)

print(f"PCA重构MSE: {pca_mse:.6f}")
print(f"自编码器重构MSE: {ae_mse:.6f}")

# 可视化重构效果对比
plt.figure(figsize=(15, 8))

for i in range(10):
    # 原始图像
    plt.subplot(3, 10, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    if i == 0:
        plt.title('原始图像', fontsize=12, y=1.2)
    plt.axis('off')
    
    # PCA重构
    plt.subplot(3, 10, i + 11)
    plt.imshow(scaler.inverse_transform(X_pca_reconstructed)[i].reshape(28, 28), cmap='gray')
    if i == 0:
        plt.title('PCA重构', fontsize=12, y=1.2)
    plt.axis('off')
    
    # 自编码器重构
    plt.subplot(3, 10, i + 21)
    plt.imshow(X_ae_reconstructed[i].reshape(28, 28), cmap='gray')
    if i == 0:
        plt.title('自编码器重构', fontsize=12, y=1.2)
    plt.axis('off')

plt.suptitle('PCA vs 自编码器重构效果对比', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

## 总结

本notebook探索了各种降维技术及其应用：

1. **PCA (主成分分析)**：
   - 是一种线性降维技术
   - 计算高效、理论成熟
   - 对于非线性关系的数据效果有限

2. **t-SNE**：
   - 非线性降维技术，保持局部结构
   - 适合数据可视化
   - 计算复杂度高，不适合大规模数据

3. **自编码器**：
   - 神经网络实现的非线性降维
   - 可以捕捉复杂的非线性关系
   - 训练需要时间，但具有较强的表达能力
   - 不同变种(基础、栈式、去噪)各有优势

每种方法都有其适用场景：
- 当数据近似线性且需要快速降维时，PCA是最佳选择
- 对于数据可视化，t-SNE通常提供最好的结果
- 当处理高度非线性数据且需要高质量重构时，自编码器是理想选择

选择降维方法时，应根据具体任务需求、数据特性以及计算资源进行权衡。